In [ ]:
# Machine Learning Results

#This notebook documents the evaluation and interpretation of the 
#Linear SVM classifier used to predict phenoconversion from slow-wave sleep biomarkers.

In [2]:
# on importe 
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path("..")

ML_RESULTS_DIR = PROJECT_ROOT / "results" / "ml"
FIGURES_DIR = PROJECT_ROOT / "figures" / "ml"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(ML_RESULTS_DIR.exists())
print(FIGURES_DIR.exists())

True
True


In [4]:
# on vérifie les fichiers produits par le script 
list(ML_RESULTS_DIR.iterdir())

[PosixPath('../results/ml/feature_importance.csv'),
 PosixPath('../results/ml/classification_metrics.csv'),
 PosixPath('../results/ml/confusion_matrix.csv')]

In [5]:
for file in ML_RESULTS_DIR.iterdir():
    print(file.name)

feature_importance.csv
classification_metrics.csv
confusion_matrix.csv


In [6]:
# on va lire les métriques 
metrics = pd.read_csv(
    ML_RESULTS_DIR / "classification_metrics.csv",
    index_col=0
)

metrics

,0
accuracy,0.687500
balanced_accuracy,0.682186
precision,0.607143
recall,0.653846
f1,0.629630


In [7]:
# je veux comprendre comment le modèle fait des erreurs 
cm = pd.read_csv(
    ML_RESULTS_DIR / "confusion_matrix.csv",
    index_col=0
)

cm

,pred_non_converter,pred_converter
true_non_converter,27,11
true_converter,9,17


In [8]:
# maintenant, je veux chercher à comprendre, quelles sont les varaibles qui perttenet au modèle de prédire la conversion 
importance = pd.read_csv(
    ML_RESULTS_DIR / "feature_importance.csv"
)

importance.head(20)

,feature,coefficient,abs_coefficient
0,occipital_total_sw_count,-1.191205,1.191205
1,occipital_total_N2_pkpk_amp_uV,0.909597,0.909597
2,frontal_total_slope_0_min,0.860606,0.860606
3,central_total_trans_freq_Hz,-0.842717,0.842717
4,parietal_total_trans_freq_Hz,-0.837975,0.837975
5,parietal_total_N2_freq_Hz,0.808701,0.808701
6,central_total_freq_Hz,-0.805231,0.805231
7,occipital_total_slope_0_min,-0.753945,0.753945
8,occipital_total_N3_pkpk_amp_uV,0.746944,0.746944
9,parietal_total_sw_count,0.744755,0.744755


In [9]:
# même chose, juste autre manière de le faire  
importance.sort_values(
    "abs_coefficient",
    ascending=False
).head(20)

,feature,coefficient,abs_coefficient
0,occipital_total_sw_count,-1.191205,1.191205
1,occipital_total_N2_pkpk_amp_uV,0.909597,0.909597
2,frontal_total_slope_0_min,0.860606,0.860606
3,central_total_trans_freq_Hz,-0.842717,0.842717
4,parietal_total_trans_freq_Hz,-0.837975,0.837975
5,parietal_total_N2_freq_Hz,0.808701,0.808701
6,central_total_freq_Hz,-0.805231,0.805231
7,occipital_total_slope_0_min,-0.753945,0.753945
8,occipital_total_N3_pkpk_amp_uV,0.746944,0.746944
9,parietal_total_sw_count,0.744755,0.744755


In [11]:
# Feature Stability Analysis
svm_importance = pd.read_csv(
    "../results/ml/feature_importance.csv"
)

log_importance = pd.read_csv(
    "../results/ml/logistic_regression_feature_importance.csv"
)

In [12]:
svm_top15 = svm_importance.head(15)

svm_top15

,feature,coefficient,abs_coefficient
0,occipital_total_sw_count,-1.191205,1.191205
1,occipital_total_N2_pkpk_amp_uV,0.909597,0.909597
2,frontal_total_slope_0_min,0.860606,0.860606
3,central_total_trans_freq_Hz,-0.842717,0.842717
4,parietal_total_trans_freq_Hz,-0.837975,0.837975
5,parietal_total_N2_freq_Hz,0.808701,0.808701
6,central_total_freq_Hz,-0.805231,0.805231
7,occipital_total_slope_0_min,-0.753945,0.753945
8,occipital_total_N3_pkpk_amp_uV,0.746944,0.746944
9,parietal_total_sw_count,0.744755,0.744755


In [13]:
log_top15 = log_importance.head(15)

log_top15

,feature,coefficient,abs_coefficient
0,occipital_total_N3_pkpk_amp_uV,0.921421,0.921421
1,central_total_trans_freq_Hz,-0.871857,0.871857
2,central_total_N3_pkpk_amp_uV,-0.713987,0.713987
3,occipital_total_N2_pkpk_amp_uV,0.676865,0.676865
4,parietal_total_N2_freq_Hz,0.671646,0.671646
5,parietal_total_sw_count,0.607254,0.607254
6,central_total_freq_Hz,-0.598979,0.598979
7,frontal_total_N3_pkpk_amp_uV,-0.589895,0.589895
8,occipital_total_N3_trans_freq_Hz,0.578748,0.578748
9,occipital_total_slope_0_min,-0.571661,0.571661


In [14]:
svm_features = set(
    svm_top15["feature"]
)

log_features = set(
    log_top15["feature"]
)

print("SVM:", len(svm_features))
print("LOG:", len(log_features))

SVM: 15
LOG: 15


In [15]:
common_features = (
    svm_features
    .intersection(log_features)
)

common_features

{'central_total_N3_pkpk_amp_uV',
 'central_total_freq_Hz',
 'central_total_trans_freq_Hz',
 'frontal_total_slope_0_min',
 'frontal_total_sw_count',
 'occipital_total_N2_pkpk_amp_uV',
 'occipital_total_N3_pkpk_amp_uV',
 'occipital_total_slope_0_min',
 'parietal_total_N2_freq_Hz',
 'parietal_total_sw_count'}

In [16]:
print(
    "Common biomarkers:",
    len(common_features)
)

Common biomarkers: 10


In [17]:
common_df = pd.DataFrame({
    "feature": sorted(
        list(common_features)
    )
})

common_df

,feature
0,central_total_N3_pkpk_amp_uV
1,central_total_freq_Hz
2,central_total_trans_freq_Hz
3,frontal_total_slope_0_min
4,frontal_total_sw_count
5,occipital_total_N2_pkpk_amp_uV
6,occipital_total_N3_pkpk_amp_uV
7,occipital_total_slope_0_min
8,parietal_total_N2_freq_Hz
9,parietal_total_sw_count


In [18]:
common_df.to_csv(
    "../results/ml/common_biomarkers.csv",
    index=False
)

In [20]:
## Robust Biomarkers

## These biomarkers were identified among the most important features by both the Linear SVM and Logistic Regression classifiers.

## These variables may represent the most robust physiological predictors of phenoconversion.

In [21]:
common_df

,feature
0,central_total_N3_pkpk_amp_uV
1,central_total_freq_Hz
2,central_total_trans_freq_Hz
3,frontal_total_slope_0_min
4,frontal_total_sw_count
5,occipital_total_N2_pkpk_amp_uV
6,occipital_total_N3_pkpk_amp_uV
7,occipital_total_slope_0_min
8,parietal_total_N2_freq_Hz
9,parietal_total_sw_count


In [22]:
# les 10 biomarqueurs les plus robustes
common_features

{'central_total_N3_pkpk_amp_uV',
 'central_total_freq_Hz',
 'central_total_trans_freq_Hz',
 'frontal_total_slope_0_min',
 'frontal_total_sw_count',
 'occipital_total_N2_pkpk_amp_uV',
 'occipital_total_N3_pkpk_amp_uV',
 'occipital_total_slope_0_min',
 'parietal_total_N2_freq_Hz',
 'parietal_total_sw_count'}

In [ ]:
# Model Comparison

#Comparison of Linear SVM and Logistic Regression classification performance.

In [23]:
# on charge les métriques 
svm_metrics = pd.read_csv(
    "../results/ml/classification_metrics.csv",
    index_col=0
)

log_metrics = pd.read_csv(
    "../results/ml/logistic_regression_metrics.csv",
    index_col=0
)

svm_metrics

,0
accuracy,0.687500
balanced_accuracy,0.682186
precision,0.607143
recall,0.653846
f1,0.629630
roc_auc,0.727733


In [24]:
# je vérifie 
log_metrics

,0
accuracy,0.578125
balanced_accuracy,0.577935
precision,0.483871
recall,0.576923
f1,0.526316
roc_auc,0.641700


In [25]:
# on construit le tableau 
comparison = pd.concat(
    [
        svm_metrics,
        log_metrics
    ],
    axis=1
)

comparison.columns = [
    "Linear_SVM",
    "Logistic_Regression"
]

comparison

,Linear_SVM,Logistic_Regression
accuracy,0.687500,0.578125
balanced_accuracy,0.682186,0.577935
precision,0.607143,0.483871
recall,0.653846,0.576923
f1,0.629630,0.526316
roc_auc,0.727733,0.641700


In [26]:
# on sauvegarde 
comparison.to_csv(
    "../results/ml/model_comparison.csv"
)

comparison

,Linear_SVM,Logistic_Regression
accuracy,0.687500,0.578125
balanced_accuracy,0.682186,0.577935
precision,0.607143,0.483871
recall,0.653846,0.576923
f1,0.629630,0.526316
roc_auc,0.727733,0.641700


In [27]:
# Regional vs Global Classification

In [28]:
# on les charge 
regional = pd.read_csv(
    "../results/ml/classification_metrics.csv",
    index_col=0
)

global_model = pd.read_csv(
    "../results/ml_global/classification_metrics.csv",
    index_col=0
)

In [29]:
# on construit le tableau 
comparison = pd.concat(
    [
        regional,
        global_model
    ],
    axis=1
)

comparison.columns = [
    "Regional_SVM",
    "Global_SVM"
]

comparison

,Regional_SVM,Global_SVM
accuracy,0.687500,0.562500
balanced_accuracy,0.682186,0.565833
precision,0.607143,0.450000
recall,0.653846,0.580645
f1,0.629630,0.507042
roc_auc,0.727733,0.549704


In [30]:
# on le sauvegarde 
comparison.to_csv(
    "../results/ml/regional_vs_global_comparison.csv"
)